# OpenCR reaction sets

[OpenCR](https://github.com/max-models/opencr) derives effective rate coefficients of
hydrogen from collisional-radiative models with resolved excited states.
`wellbalanced.cr` turns them into `Reaction` objects for the global and fluid models, in
place of ground-state cross sections and placeholders. It needs the `cr` extra
(`pip install "wellbalanced[cr]"`).

In [ ]:
from wellbalanced.cr import CR_TEMPERATURES, opencr_hydrogen_reactions

reactions = opencr_hydrogen_reactions(electron_density=1.0e17)
[r.name for r in reactions]

The effective rates depend on the electron density as well as the temperature; a
`Reaction` depends on the temperature only, so they are taken at `electron_density`, which
should be near the density of the solution (below about 1e18 m⁻³ the excited states are
coronal and the dependence is weak). The energy each electron loses per H, H+ and H2,
radiation included, is a pair of reactions that change no density.

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr

temperature = xr.DataArray(
    CR_TEMPERATURES,
    dims="te",
    attrs={"long_name": "electron temperature", "units": "eV"},
)
shown = ["CR ionization of H", "ionization of H2", "dissociation of H2",
         "CR recombination of H+"]  # fmt: skip
rates = xr.DataArray(
    np.array(
        [
            [
                next(r for r in reactions if r.name == name).coefficient(t, 0.03)
                for t in CR_TEMPERATURES
            ]
            for name in shown
        ]
    ),  # fmt: skip
    coords={"reaction": shown, "te": temperature},
    name="rate",
    attrs={"long_name": "rate coefficient", "units": "m^3/s"},
)
with plasma_plots.figure(1, 1, figsize=(6, 4)) as fig:
    rates.plasma.plot.profiles(x="te", over="reaction", logx=True, logy=True, ax=fig[0])

## A driver with OpenCR's rates

`hydrogen_driver` is the 0D RF driver of the [RF driver tutorial](/wellbalanced/tutorials/rf-driver/)
with these reactions; `deuterium_neutralizer` does the same for the
[neutralizer](/wellbalanced/tutorials/neutralizer/). Against the placeholder rates at
25 kW absorbed:

In [ ]:
import warnings

from wellbalanced.cr import hydrogen_driver
from wellbalanced.driver import illustrative_hydrogen_driver

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)  # the placeholder-data warning
    drivers = {
        "placeholders": illustrative_hydrogen_driver(),
        "OpenCR": hydrogen_driver(),
    }
for name, driver in drivers.items():
    s = driver.solve(25e3)
    print(
        f"{name:>12}: n_e = {s.plasma.electron_density:.2e} m^-3, "
        f"T_e = {s.plasma.electron_temperature:.2f} eV, efficiency {s.efficiency:.2f}"
    )

The dissociation now heats the gas (`gas_heating`, a few eV per event from the
Franck-Condon fragments), which a `GasTemperature` balance takes up, see
[Sources and gas heating](/wellbalanced/tutorials/sources-and-gas/).